In [1]:
import pandas as pd
s = pd.Series([25.0, 30.5, 22.0], index=['sensor_1','sensor_2','sensor_3'])
df = pd.DataFrame({'device' : ['A1', 'B2', 'C3'],'temp_c' : [36.5, 41.2, 38.9],'active': [ True, False, True]})
print(df.dtypes)

device     object
temp_c    float64
active       bool
dtype: object


In [2]:
df = pd.DataFrame({'age' : [ 19 ,24, 35, 42],'score' : [88, 92, 79, 95]}, index= [ 'u1', 'u2', 'u3', 'u4'])
sub1 = df.loc['u1':'u3', ['score']]
sub2 = df.iloc[0:2, 1]
high_performers = df[(df['age'] > 20) & (df['score']>= 90)]
print(high_performers)

    age  score
u2   24     92
u4   42     95


In [3]:
import numpy as np
df = pd.DataFrame({'reading' : [ 10.2 ,np.nan, 15.8, np.nan, 20.1], 'status' : [ 'OK', 'WARN', np.nan, 'OK', 'OK']})
null_counts = df.isna().sum()
df['reading_ffill'] = df['reading'].ffill()
mean_val = df['reading'].mean()
df['reading_imputed'] = df['reading'].fillna(mean_val)
print(df[['reading','reading_imputed']])

   reading  reading_imputed
0     10.2        10.200000
1      NaN        15.366667
2     15.8        15.800000
3      NaN        15.366667
4     20.1        20.100000


In [4]:
users = pd.DataFrame({'uid': [101,102,103], 'name' : ['Alice', 'Bob', 'Charlie']})
orders = pd.DataFrame({ 'order_id':['0.1','0.2', '0.3'],'user_id':[101,101,104], 'amount': [45.0, 120.0, 85.0]})
merged = pd.merge(users, orders, left_on='uid' , right_on='user_id', how='inner')
print(merged[['uid', 'name', 'order_id', 'amount']])

   uid   name order_id  amount
0  101  Alice      0.1    45.0
1  101  Alice      0.2   120.0


In [5]:
b1 = pd.DataFrame({'val': [10,20]})
b2 = pd.DataFrame({'val': [30,40]})
combined = pd.concat([b1, b2] , keys=['batch1', 'batch2'])
primary = pd.Series([np.nan, 2.5, np.nan, 4.0])
fallback = pd.Series([1.0, 9.9, 3.0, 9.9])
reconciled = primary.combine_first(fallback)
print(reconciled.tolist())

[1.0, 2.5, 3.0, 4.0]


In [6]:
wide = pd.DataFrame({ 'date' : ['2025-01', '2025-02'], 'sensor_A': [21.5, 22.0], 'sensor_B': [19.0, 19.8]})
long_df = pd.melt(wide, id_vars=['date'], var_name='sensor', value_name='temp')
pivoted= long_df.pivot_table(index='date', columns= 'sensor',values='temp', aggfunc='mean')
print(pivoted)

sensor   sensor_A  sensor_B
date                       
2025-01      21.5      19.0
2025-02      22.0      19.8


In [7]:
sales = pd.DataFrame({ 'region': ['North', 'North', 'South', 'South', 'North'], 'product' : ['A', 'B', 'A','B','A'], 'revenue': [120,80,200,150,140]})
summary = sales.groupby('region').agg(total_rev=('revenue','sum'),avg_rev=('revenue', 'mean'),tx_count=('revenue','count'))
print(summary)

        total_rev     avg_rev  tx_count
region                                 
North         340  113.333333         3
South         350  175.000000         2


In [8]:
df = pd.DataFrame({'dept': ['IT', 'IT', 'IT', 'HR', 'HR'], 'salary': [70000,90000,80000,50000,60000]})
dept_avg = df.groupby('dept')['salary'].transform('mean')
df['salary_vs_dept'] = df['salary'] - dept_avg
zscore = lambda x: (x - x.mean()) / x.std()
df['z_score'] = df.groupby('dept')['salary'].transform(zscore)
print(df[['dept', 'salary', 'salary_vs_dept']])

  dept  salary  salary_vs_dept
0   IT   70000        -10000.0
1   IT   90000         10000.0
2   IT   80000             0.0
3   HR   50000         -5000.0
4   HR   60000          5000.0


In [9]:
ratings = pd.Series(['med', 'low', 'high', 'low', 'high'] * 1000)
mem_raw = ratings.memory_usage(deep=True)
cat_type = pd.CategoricalDtype( categories= ['low', 'med', 'high'], ordered=True)
cat_ratings = ratings.astype(cat_type)
mem_cat = cat_ratings.memory_usage(deep=True)
print(f"Memory: {mem_raw}B -> {mem_cat}B")
print(cat_ratings.min())

Memory: 262132B -> 5397B
low


In [10]:
arrays = [['Store_1', 'Store_1', 'Store_2', 'Store_2'], [2024,2025,2024,2025]]
idx = pd.MultiIndex.from_arrays(arrays, names = ['store', 'year'])
df=pd.DataFrame({'sales': [100,150,80, 120]}, index=idx)
s1 = df.xs('Store_1', level= 'store')
wide_panel = df.unstack(level='year')
print(wide_panel)

        sales     
year     2024 2025
store             
Store_1   100  150
Store_2    80  120


In [11]:
raw = pd.DataFrame({ 'age' : [17, 25, 38, 55, 16], 'income': [0, 45000, 82000, 120000, 500]})
def standardize_income(df_in): return df_in.assign(z_inc=lambda d: (d.income - d.income.mean()) / d.income.std())
processed = ( raw
              .query('age >= 18')
              .assign(log_inc=lambda d: np.log1p(d['income']))
              .pipe(standardize_income)
              .sort_values(by='income', ascending=False))
print(processed[['age', 'income', 'log_inc']])

   age  income    log_inc
3   55  120000  11.695255
2   38   82000  11.314487
1   25   45000  10.714440
